In [3]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import average_precision_score, roc_auc_score

def select_side_on_finalmag_linear_both_modalities(
    dataset_names: List[str],
    *,
    suffix: str = "_finalmag.csv",
    id_cols = ("Patient","Session","Task"),
    label_col: str = "FOG",
    window_size: int = 80,
    overlap: float = 0.5,
    require_both_modalities: bool = False,   # True -> score only windows where both acc+gyro exist for that side
    min_windows_per_side: int = 100,         # minimum windows to trust a side’s score
    prefer_side: str = "l",                  # tie-break preference
    pos_weight: float = None                 # None -> balanced weights; else custom positive weight multiplier
) -> pd.DataFrame:
    """
    For each dataset and each bilateral stem (e.g., foot, ankle, thigh...):
      - build per-window features for each side (L/R) by concatenating simple stats from acc and gyro:
        [acc_mean_v, acc_std_v, acc_mean_mag, acc_std_mag, gyro_mean_v, gyro_std_v, gyro_mean_mag, gyro_std_mag]
        (if a modality is missing, only the available part is used)
      - label each window by majority (>0.5) FOG within the window (no crossing registration boundaries)
      - fit LinearRegression with balanced sample weights; score with K-fold AUPRC (fallback AUROC)
      - choose the better side; tie -> more windows; tie -> prefer_side
    Returns a tidy DataFrame with one row per (dataset, stem) decision.
    """

    prefer_side = prefer_side.lower()
    assert prefer_side in ("l", "r")
    step = max(1, int(round(window_size * (1.0 - overlap))))

    # flexible column parser
    pat_feat = re.compile(r"^(?P<base>.+?)_(?P<mod>acc|gyro)_(?P<feat>v|mag)$", re.IGNORECASE)
    pat_side = re.compile(r"^(?P<stem>.+)_(?P<side>l|r)$", re.IGNORECASE)

    def _list_stems_with_sides(cols):
        """Return mapping stem -> sides/modalities present in this dataset."""
        # collect all base,mod,feat
        parsed = []
        for c in cols:
            m = pat_feat.match(str(c).strip())
            if m:
                base, mod, feat = m.group("base"), m.group("mod").lower(), m.group("feat").lower()
                parsed.append((base, mod, feat))
        if not parsed:
            return {}

        # find side in base if present
        stems = {}  # stem -> {'l': {'acc': bool, 'gyro': bool}, 'r': {...}}
        for base, mod, feat in parsed:
            ms = pat_side.match(base)
            if not ms:
                continue
            stem = ms.group("stem").lower()
            side = ms.group("side").lower()
            stems.setdefault(stem, {'l': {'acc': False, 'gyro': False}, 'r': {'acc': False, 'gyro': False}})
            stems[stem][side][mod] = True
        # keep only stems that have at least one modality per side (so both L and R exist in some form)
        stems = {
            stem: sides for stem, sides in stems.items()
            if (sides['l']['acc'] or sides['l']['gyro']) and (sides['r']['acc'] or sides['r']['gyro'])
        }
        return stems

    def _window_iter(group_len):
        """Yield (start,end) indices for full windows within a registration."""
        for start in range(0, group_len - window_size + 1, step):
            yield start, start + window_size

    def _majority_label(yseg):
        frac = float(np.mean(yseg))
        if frac > 0.5: return 1
        if frac < 0.5: return 0
        # exact tie -> prefer negative
        return 0

    def _side_feats_for_segment(seg: pd.DataFrame, stem: str, side: str, want_acc: bool, want_gyro: bool):
        feats = []
        # acc
        if want_acc:
            pv = f"{stem}_{side}_acc_v"; pm = f"{stem}_{side}_acc_mag"
            if pv in seg and pm in seg:
                v = seg[pv].to_numpy(dtype=float); m = seg[pm].to_numpy(dtype=float)
                feats += [np.nanmean(v), np.nanstd(v), np.nanmean(m), np.nanstd(m)]
        # gyro
        if want_gyro:
            pv = f"{stem}_{side}_gyro_v"; pm = f"{stem}_{side}_gyro_mag"
            if pv in seg and pm in seg:
                v = seg[pv].to_numpy(dtype=float); m = seg[pm].to_numpy(dtype=float)
                feats += [np.nanmean(v), np.nanstd(v), np.nanmean(m), np.nanstd(m)]
        return feats

    def _cv_score_linear(F: np.ndarray, y: np.ndarray, w: np.ndarray, folds: int = 5, seed: int = 42) -> float:
        """LinearRegression -> predict continuous -> AUPRC (fallback AUROC)."""
        if F.shape[0] < 10 or len(np.unique(y)) < 2 or F.shape[1] == 0:
            return np.nan
        counts = np.unique(y, return_counts=True)[1]
        k = int(min(folds, counts.min())) if counts.min() > 1 else 1
        if k < 2:
            return np.nan
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        scores = []
        for tr, va in skf.split(F, y):
            Xtr, ytr, wtr = F[tr], y[tr], w[tr]
            Xva, yva = F[va], y[va]
            if len(np.unique(ytr)) < 2 or len(np.unique(yva)) < 2:
                continue
            lr = LinearRegression()
            lr.fit(Xtr, ytr, sample_weight=wtr)
            p = lr.predict(Xva)
            try:
                s = average_precision_score(yva, p)
            except Exception:
                try: s = roc_auc_score(yva, p)
                except Exception: s = np.nan
            if not np.isnan(s):
                scores.append(float(s))
        return float(np.mean(scores)) if scores else np.nan

    rows = []

    for ds in dataset_names:
        path = Path(f"{ds}{suffix}")
        if not path.exists():
            print(f"⚠️ missing file: {path}")
            continue

        df = pd.read_csv(path)
        # checks
        if any(c not in df.columns for c in id_cols + (label_col,)):
            print(f"⚠️ {ds} missing required columns among {id_cols + (label_col,)}")
            continue

        stems = _list_stems_with_sides(df.columns)
        if not stems:
            continue

        # build windows respecting registrations
        X_windows = []   # we’ll build features per stem-side later, so just keep segmentation indices
        y_windows = []
        reg_offsets = []  # (reg_start_idx, reg_len) for mapping if needed
        for key, g in df.groupby(list(id_cols), sort=False):
            g = g.reset_index(drop=True)
            L = len(g)
            if L < window_size:
                continue
            for start, end in _window_iter(L):
                seg = g.iloc[start:end]
                y_windows.append(_majority_label(seg[label_col].to_numpy(dtype=float)))
                # store indices relative to full df for later feature extraction
                X_windows.append((seg.index[0] + g.index[0], seg.index[-1] + g.index[0]))  # absolute index range

        if not X_windows or len(np.unique(y_windows)) < 2:
            continue

        yW = np.asarray(y_windows, dtype=int)
        # balanced sample weights by default
        pos_ratio = yW.mean()
        if pos_weight is None:
            w_pos = (1 - pos_ratio) / max(pos_ratio, 1e-8) if 0 < pos_ratio < 1 else 1.0
        else:
            w_pos = float(pos_weight)
        weights = np.where(yW == 1, w_pos, 1.0).astype(np.float32)

        # For each stem: evaluate L vs R using both modalities together (as available)
        for stem, sides in stems.items():
            # modality availability per side in this dataset
            L_has_acc, L_has_gyro = sides['l']['acc'], sides['l']['gyro']
            R_has_acc, R_has_gyro = sides['r']['acc'], sides['r']['gyro']

            # enforce modality requirement
            if require_both_modalities:
                if not (L_has_acc and L_has_gyro) or not (R_has_acc and R_has_gyro):
                    # if either side misses a modality, skip this stem
                    continue
            else:
                if not (L_has_acc or L_has_gyro) or not (R_has_acc or R_has_gyro):
                    continue

            # build feature matrices per side
            FL, FR = [], []
            for (abs_start, abs_end), yw in zip(X_windows, yW):
                seg = df.iloc[abs_start:abs_end]
                fL = _side_feats_for_segment(seg, stem, 'l', L_has_acc, L_has_gyro)
                fR = _side_feats_for_segment(seg, stem, 'r', R_has_acc, R_has_gyro)
                # keep only if features are non-empty for that side
                if fL:
                    FL.append(fL)
                else:
                    FL.append([np.nan])  # mark missing
                if fR:
                    FR.append(fR)
                else:
                    FR.append([np.nan])

            # filter out windows where side features missing
            FL = [f for f in FL if not (len(f)==1 and np.isnan(f[0]))]
            FR = [f for f in FR if not (len(f)==1 and np.isnan(f[0]))]
            # align labels/weights to the kept windows for each side
            idxL = [i for i,(abs_start, abs_end) in enumerate(X_windows) if not (len(_side_feats_for_segment(df.iloc[abs_start:abs_end], stem, 'l', L_has_acc, L_has_gyro))==0)]
            idxR = [i for i,(abs_start, abs_end) in enumerate(X_windows) if not (len(_side_feats_for_segment(df.iloc[abs_start:abs_end], stem, 'r', R_has_acc, R_has_gyro))==0)]

            nL, nR = len(idxL), len(idxR)
            scoreL = scoreR = np.nan

            if nL >= min_windows_per_side and len(np.unique(yW[idxL])) >= 2:
                FL = np.asarray(FL, dtype=np.float32)
                yL = yW[idxL]
                wL = weights[idxL]
                scoreL = _cv_score_linear(FL, yL, wL)

            if nR >= min_windows_per_side and len(np.unique(yW[idxR])) >= 2:
                FR = np.asarray(FR, dtype=np.float32)
                yR = yW[idxR]
                wR = weights[idxR]
                scoreR = _cv_score_linear(FR, yR, wR)

            # choose side
            if np.isnan(scoreL) and np.isnan(scoreR):
                if nL > nR:
                    chosen, reason = "l", "more_windows"
                elif nR > nL:
                    chosen, reason = "r", "more_windows"
                else:
                    chosen, reason = prefer_side, f"tie_windows_prefer_{prefer_side}"
            else:
                sL = -1.0 if np.isnan(scoreL) else scoreL
                sR = -1.0 if np.isnan(scoreR) else scoreR
                if sL > sR:
                    chosen, reason = "l", "higher_score"
                elif sR > sL:
                    chosen, reason = "r", "higher_score"
                else:
                    if nL > nR:
                        chosen, reason = "l", "tie_score_more_windows"
                    elif nR > nL:
                        chosen, reason = "r", "tie_score_more_windows"
                    else:
                        chosen, reason = prefer_side, f"tie_score_windows_prefer_{prefer_side}"

            rows.append({
                "dataset": ds,
                "stem": stem,
                "score_left": None if np.isnan(scoreL) else float(scoreL),
                "score_right": None if np.isnan(scoreR) else float(scoreR),
                "chosen": chosen,
                "reason": reason
            })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["dataset","stem"]).reset_index(drop=True)
    return out


In [4]:
os.chdir('Structured_datasets')
datasets = ['FOGSTAR', 'TDCS', 'IMU','Multimodal','Oday', 'rempark', 'Daphnet']

side_decisions = select_side_on_finalmag_linear_both_modalities(
    datasets,
    window_size=80,
    overlap=0.5,
    require_both_modalities=False,  # set True if you only trust windows where both acc+gyro exist
    min_windows_per_side=100,
    prefer_side="l",
    pos_weight=None                  # None = balanced automatically
)

display(side_decisions)
os.chdir('..')

,dataset,stem,score_left,score_right,chosen,reason
0,FOGSTAR,ankle,0.392795,0.397077,r,higher_score
1,Multimodal,shank,0.563937,0.450589,l,higher_score
2,Oday,ankle,0.282088,0.282759,r,higher_score
3,Oday,foot,0.281869,0.266216,l,higher_score
4,Oday,thigh,0.263291,0.279246,r,higher_score
5,Oday,wrist,0.289036,0.292253,r,higher_score


In [8]:
import re
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Union

# "{base}_{acc|gyro}_{v|mag}"
PAT = re.compile(r"^(?P<base>.+?)_(?P<mod>acc|gyro)_(?P<feat>v|mag)$", re.IGNORECASE)
# base like "{stem}_{l|r}"
SIDE_PAT = re.compile(r"^(?P<stem>.+)_(?P<side>l|r)$", re.IGNORECASE)

def clean_finalmag_bilateral_then_assimilate(
    input_path: Union[str, Path],
    side_decisions_df: pd.DataFrame,
    *,
    dataset_name: Optional[str] = None,
    overwrite: bool = True,
    output_path: Optional[Union[str, Path]] = None,

    # Phase 2: assimilation rules (applied after bilaterality)
    replace_prefix_map: Dict[str, str] = None,  # e.g. {"lower_back":"lowerback","arm":"wrist","waist":"lowerback"}
    drop_stems: List[str] = None,               # e.g. ["foot","chest","head"]
):
    """
    Phase 1 (bilaterality first):
      - drop not-chosen side columns
      - remove '_l|_r' from chosen side base => one-sided canonical sensor name

    Phase 2 (assimilation second):
      - apply prefix replacements on base (e.g., waist->lowerback)
      - drop specified stems (foot/chest/head)

    Writes cleaned CSV (overwrite by default). Returns output path.
    """
    input_path = Path(input_path)

    if dataset_name is None:
        dataset_name = input_path.name.replace("_finalmag.csv", "")

    if replace_prefix_map is None:
        replace_prefix_map = {
            "lower_back": "lowerback",
            "arm": "wrist",
            "waist": "lowerback",
            "shank": "ankle",
        }
    if drop_stems is None:
        drop_stems = ["foot", "chest", "head"]
    drop_set = {s.lower() for s in drop_stems}

    # output
    if output_path is None:
        out_path = input_path if overwrite else input_path.with_name(input_path.stem + "_cleaned.csv")
    else:
        out_path = Path(output_path)

    df = pd.read_csv(input_path)

    # ----------------------------
    # Build stem -> chosen side map for THIS dataset
    # side_decisions_df columns expected: dataset, stem, chosen
    # ----------------------------
    dec = side_decisions_df.copy()
    dec = dec[dec["dataset"].astype(str).str.lower() == str(dataset_name).lower()]
    stem2side = {
        str(r["stem"]).lower(): str(r["chosen"]).lower()
        for _, r in dec.iterrows()
        if str(r.get("chosen","")).lower() in ("l","r")
    }

    # ============================================================
    # PHASE 1 — handle bilaterality (drop other side + strip side tag)
    # ============================================================
    drop_cols = []
    rename_phase1 = {}

    for col in df.columns:
        m = PAT.match(col)
        if not m:
            continue

        base, mod, feat = m.group("base"), m.group("mod"), m.group("feat")
        ms = SIDE_PAT.match(base)
        if not ms:
            continue  # unsided column: keep for now

        stem = ms.group("stem").lower()
        side = ms.group("side").lower()

        if stem not in stem2side:
            # no decision for this stem => keep as-is for now
            continue

        chosen = stem2side[stem]
        if side != chosen:
            drop_cols.append(col)
        else:
            # keep chosen side but remove _l/_r => one-sided name
            rename_phase1[col] = f"{stem}_{mod}_{feat}"

    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")
    if rename_phase1:
        df = df.rename(columns=rename_phase1)

    # ============================================================
    # PHASE 2 — assimilation + drop stems SECOND
    # ============================================================
    rename_phase2 = {}
    drop_cols2 = []

    for col in df.columns:
        m = PAT.match(col)
        if not m:
            continue

        base, mod, feat = m.group("base"), m.group("mod"), m.group("feat")
        base_l = base.lower()

        # drop stems (foot/chest/head) AFTER bilaterality
        stem0 = base_l.split("_")[0]
        if stem0 in drop_set:
            drop_cols2.append(col)
            continue

        # apply assimilation (prefix replacements) AFTER bilaterality
        new_base = base
        for old, new in replace_prefix_map.items():
            if new_base.startswith(old):
                new_base = new_base.replace(old, new, 1)

        if new_base != base:
            rename_phase2[col] = f"{new_base}_{mod}_{feat}"

    if drop_cols2:
        df = df.drop(columns=drop_cols2, errors="ignore")
    if rename_phase2:
        df = df.rename(columns=rename_phase2)

    df.to_csv(out_path, index=False)
    return out_path
import re
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Union

# "{base}_{acc|gyro}_{v|mag}"
PAT = re.compile(r"^(?P<base>.+?)_(?P<mod>acc|gyro)_(?P<feat>v|mag)$", re.IGNORECASE)
# base like "{stem}_{l|r}"
SIDE_PAT = re.compile(r"^(?P<stem>.+)_(?P<side>l|r)$", re.IGNORECASE)

def clean_finalmag_bilateral_then_assimilate(
    input_path: Union[str, Path],
    side_decisions_df: pd.DataFrame,
    *,
    dataset_name: Optional[str] = None,
    overwrite: bool = True,
    output_path: Optional[Union[str, Path]] = None,

    # Phase 2: assimilation rules (applied after bilaterality)
    replace_prefix_map: Dict[str, str] = None,  # e.g. {"lower_back":"lowerback","arm":"wrist","waist":"lowerback"}
    drop_stems: List[str] = None,               # e.g. ["foot","chest","head"]
):
    """
    Phase 1 (bilaterality first):
      - drop not-chosen side columns
      - remove '_l|_r' from chosen side base => one-sided canonical sensor name

    Phase 2 (assimilation second):
      - apply prefix replacements on base (e.g., waist->lowerback)
      - drop specified stems (foot/chest/head)

    Writes cleaned CSV (overwrite by default). Returns output path.
    """
    input_path = Path(input_path)

    if dataset_name is None:
        dataset_name = input_path.name.replace("_finalmag.csv", "")

    if replace_prefix_map is None:
        replace_prefix_map = {
            "lower_back": "lowerback",
            "arm": "wrist",
            "waist": "lowerback",
        }
    if drop_stems is None:
        drop_stems = ["foot", "chest", "head"]
    drop_set = {s.lower() for s in drop_stems}

    # output
    if output_path is None:
        out_path = input_path if overwrite else input_path.with_name(input_path.stem + "_inputMOE.csv")
    else:
        out_path = Path(output_path)

    df = pd.read_csv(input_path)

    # ----------------------------
    # Build stem -> chosen side map for THIS dataset
    # side_decisions_df columns expected: dataset, stem, chosen
    # ----------------------------
    dec = side_decisions_df.copy()
    dec = dec[dec["dataset"].astype(str).str.lower() == str(dataset_name).lower()]
    stem2side = {
        str(r["stem"]).lower(): str(r["chosen"]).lower()
        for _, r in dec.iterrows()
        if str(r.get("chosen","")).lower() in ("l","r")
    }

    # ============================================================
    # PHASE 1 — handle bilaterality (drop other side + strip side tag)
    # ============================================================
    drop_cols = []
    rename_phase1 = {}

    for col in df.columns:
        m = PAT.match(col)
        if not m:
            continue

        base, mod, feat = m.group("base"), m.group("mod"), m.group("feat")
        ms = SIDE_PAT.match(base)
        if not ms:
            continue  # unsided column: keep for now

        stem = ms.group("stem").lower()
        side = ms.group("side").lower()

        if stem not in stem2side:
            # no decision for this stem => keep as-is for now
            continue

        chosen = stem2side[stem]
        if side != chosen:
            drop_cols.append(col)
        else:
            # keep chosen side but remove _l/_r => one-sided name
            rename_phase1[col] = f"{stem}_{mod}_{feat}"

    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")
    if rename_phase1:
        df = df.rename(columns=rename_phase1)

    # ============================================================
    # PHASE 2 — assimilation + drop stems 
    # ============================================================
    rename_phase2 = {}
    drop_cols2 = []

    for col in df.columns:
        m = PAT.match(col)
        if not m:
            continue

        base, mod, feat = m.group("base"), m.group("mod"), m.group("feat")
        base_l = base.lower()

        # drop stems (foot/chest/head)
        stem0 = base_l.split("_")[0]
        if stem0 in drop_set:
            drop_cols2.append(col)
            continue

        # apply assimilation (prefix replacements) AFTER bilaterality
        new_base = base
        for old, new in replace_prefix_map.items():
            if new_base.startswith(old):
                new_base = new_base.replace(old, new, 1)

        if new_base != base:
            rename_phase2[col] = f"{new_base}_{mod}_{feat}"

    if drop_cols2:
        df = df.drop(columns=drop_cols2, errors="ignore")
    if rename_phase2:
        df = df.rename(columns=rename_phase2)

    df.to_csv(out_path, index=False, mode = 'w')
    return out_path


In [9]:
from pathlib import Path

datasets = ['FOGSTAR', 'TDCS', 'IMU','Multimodal','Oday', 'rempark', 'Daphnet']

for ds in datasets:
    fp = Path("Structured_datasets") / f"{ds}_finalmag.csv"
    if not fp.exists():
        print("missing:", fp)
        continue

    out = clean_finalmag_bilateral_then_assimilate(
        fp,
        side_decisions_df=side_decisions,  # <- your decisions table
        dataset_name=ds,
        overwrite=False,
        replace_prefix_map={"lower_back":"lowerback","arm":"wrist","waist":"lowerback", 'shank':'ankle'},
        drop_stems=["foot","chest","head"],
    )
    print("✅ wrote:", out)


✅ wrote: Structured_datasets/FOGSTAR_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/TDCS_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/IMU_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/Multimodal_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/Oday_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/rempark_finalmag_inputMOE.csv
✅ wrote: Structured_datasets/Daphnet_finalmag_inputMOE.csv
